# CAR PRICE PREDICTION - DATA PREPROCESSING PIPELINE

## Complete Data Processing Workflow

This notebook follows a systematic approach to prepare data for machine learning:

```
Raw Data (has text)
    ↓
One-Hot Encode (convert text to 0/1)
    ↓
Drop Car ID & Price
    ↓
ALL DATA IS NOW NUMERIC (float64 or int64)
    ↓
Split into train/test
    ↓
THEN Scale
    ↓
Ready for Training
```

---

1. Load Data & EDA (DONE ✓)
   ↓
2. Handle Missing Values (median/mode)
   ↓
3. Remove Duplicates
   ↓
4. Drop Car ID
   ↓
5. Feature Engineering (create Age, Mileage_per_Year, etc.)
   ↓
6. Handle Outliers (IQR method)
   ↓
7. Encode Categorical Variables (Label Encoding for trees)
   ↓
8. Train-Test Split (80-20)
   ↓
9. Train Baseline Models:
   - Linear Regression (quick baseline)
   - Ridge Regression
   ↓
10. Train Tree-Based Models:
    - Random Forest ⭐
    - XGBoost ⭐
    - LightGBM ⭐
   ↓
11. Compare Models (R², RMSE, MAE)
   ↓
12. Hyperparameter Tune Best 2-3 Models
   ↓
13. Final Evaluation on Test Set
   ↓
14. Feature Importance Analysis
   ↓
15. Save Best Model

In [960]:
import pandas as pd
import numpy as np 
import math as m
import sklearn as sk
import matplotlib.pyplot as plt
from copy import copy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [961]:
data_set=pd.read_csv("data_set.csv")

In [962]:
print(f"data_set.info(): {data_set.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Car ID        2250 non-null   float64
 1   Brand         2250 non-null   object 
 2   Year          2250 non-null   float64
 3   Engine Size   2250 non-null   float64
 4   Fuel Type     2250 non-null   object 
 5   Transmission  2250 non-null   object 
 6   Mileage       2250 non-null   float64
 7   Condition     2250 non-null   object 
 8   Price         2250 non-null   float64
 9   Model         2250 non-null   object 
dtypes: float64(5), object(5)
memory usage: 195.4+ KB
data_set.info(): None


In [963]:
data_set.head()

,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1.0,Tesla,2016.0,2.3,Petrol,Manual,114832.0,New,26613.92,Model X
1,2.0,BMW,2018.0,4.4,Electric,Manual,143190.0,Used,14679.61,5 Series
2,3.0,Audi,2013.0,4.5,Electric,Manual,181601.0,New,44402.61,A4
3,4.0,Tesla,2011.0,4.1,Diesel,Automatic,68682.0,New,86374.33,Model Y
4,5.0,Ford,2009.0,2.6,Diesel,Manual,223009.0,Like New,73577.10,Mustang


In [964]:
data_set.isnull().sum()

Car ID          250
Brand           250
Year            250
Engine Size     250
Fuel Type       250
Transmission    250
Mileage         250
Condition       250
Price           250
Model           250
dtype: int64

In [965]:
data_set.dropna(inplace=True)

In [966]:
data_set.duplicated().sum()

np.int64(0)

In [967]:
data_set.drop_duplicates(inplace=True)

In [968]:
print(f"the shape of data set = {data_set.shape} and type is {type(data_set)}")

the shape of data set = (2250, 10) and type is <class 'pandas.core.frame.DataFrame'>


# encoding the non-numeric data

In [969]:
# First, see what unique values are actually there
print(data_set['Condition'].unique())
print(data_set['Condition'].value_counts())

['New' 'Used' 'Like New']
Condition
Used        769
Like New    746
New         735
Name: count, dtype: int64


# seprating text and numeric data

In [970]:
# Check data types and identify non-numeric columns
print(data_set.dtypes)
print("\nthe non numeric colums are below")
text_ds=data_set.select_dtypes(exclude=[np.number]).columns.tolist()
print(text_ds)
numeric_ds=data_set.select_dtypes(include=[np.number]).columns.tolist()
print("\nthe numeric colums are below")

print(numeric_ds)

Car ID          float64
Brand            object
Year            float64
Engine Size     float64
Fuel Type        object
Transmission     object
Mileage         float64
Condition        object
Price           float64
Model            object
dtype: object

the non numeric colums are below
['Brand', 'Fuel Type', 'Transmission', 'Condition', 'Model']

the numeric colums are below
['Car ID', 'Year', 'Engine Size', 'Mileage', 'Price']


In [971]:
# we will now laBel the text data set using one hot encoding and use condition
#mapping for the condition of car column
condition_mapping = {
    'New':0  ,
    'Like New':1,
    'Used':2
}

In [972]:
#applying the mapping to the 'Condition' column
data_set['Condition'] = data_set['Condition'].map(condition_mapping)
print(f"the non numeric data set is below \n {data_set[text_ds].head()}")

the non numeric data set is below 
    Brand Fuel Type Transmission  Condition     Model
0  Tesla    Petrol       Manual          0   Model X
1    BMW  Electric       Manual          2  5 Series
2   Audi  Electric       Manual          0        A4
3  Tesla    Diesel    Automatic          0   Model Y
4   Ford    Diesel       Manual          1   Mustang


In [973]:
#one hot encoding for the 'non numeric' column
# now we will one hot encode the text data set and drop the original text columns
#for safe execution we will create a copy of data set and then do the one hot encoding
data_set_copied=copy(data_set[text_ds].drop(columns='Condition'))
print(f"the copied data set is below \n {data_set_copied.head()}")

the copied data set is below 
    Brand Fuel Type Transmission     Model
0  Tesla    Petrol       Manual   Model X
1    BMW  Electric       Manual  5 Series
2   Audi  Electric       Manual        A4
3  Tesla    Diesel    Automatic   Model Y
4   Ford    Diesel       Manual   Mustang


In [974]:
#now we will perfom hot encoding on data_set_copied and applying condition mapping
#is not a good idea as we donot have a criteria like diesel>petrol>cng>hyBrid
print(data_set_copied["Fuel Type"].unique())
print(data_set_copied["Fuel Type"].value_counts())

['Petrol' 'Electric' 'Diesel' 'Hybrid']
Fuel Type
Diesel      587
Petrol      561
Electric    554
Hybrid      548
Name: count, dtype: int64


# One Hot Encoding 


In [975]:
data_set_copied=pd.get_dummies(data_set_copied ,drop_first=False)

data_set_copied.replace({True:1,False:0},inplace=True)
print(f"the one hot encoded data set is below \n {data_set_copied.head()}")

the one hot encoded data set is below 
    Brand_Audi  Brand_BMW  Brand_Ford  Brand_Honda  Brand_Mercedes  \
0           0          0           0            0               0   
1           0          1           0            0               0   
2           1          0           0            0               0   
3           0          0           0            0               0   
4           0          0           1            0               0   

   Brand_Tesla  Brand_Toyota  Fuel Type_Diesel  Fuel Type_Electric  \
0            1             0                 0                   0   
1            0             0                 0                   1   
2            0             0                 0                   1   
3            1             0                 1                   0   
4            0             0                 1                   0   

   Fuel Type_Hybrid  ...  Model_Model S  Model_Model X  Model_Model Y  \
0                 0  ...              0            

C:\Users\Admin\AppData\Local\Temp\ipykernel_11548\1894895654.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_set_copied.replace({True:1,False:0},inplace=True)


# Combining the all data 

In [976]:
#concatenating the one hot encoded data set with the numeric data set to get the final data set for training

train_ds= pd.concat([data_set[numeric_ds],data_set_copied],axis=1)
print(f"the final data set for training is below \n {train_ds.head()}")

the final data set for training is below 
    Car ID    Year  Engine Size   Mileage     Price  Brand_Audi  Brand_BMW  \
0     1.0  2016.0          2.3  114832.0  26613.92           0          0   
1     2.0  2018.0          4.4  143190.0  14679.61           0          1   
2     3.0  2013.0          4.5  181601.0  44402.61           1          0   
3     4.0  2011.0          4.1   68682.0  86374.33           0          0   
4     5.0  2009.0          2.6  223009.0  73577.10           0          0   

   Brand_Ford  Brand_Honda  Brand_Mercedes  ...  Model_Model S  Model_Model X  \
0           0            0               0  ...              0              1   
1           0            0               0  ...              0              0   
2           0            0               0  ...              0              0   
3           0            0               0  ...              0              0   
4           1            0               0  ...              0              0   

   Mode

we have everything about data now; we have to do feature engineering, ,caling and plotting.

using feature scaling for numeric columns to shorten the values 

# Seprating the X & Y

In [977]:
# CORRECT:
X = train_ds.drop(columns=['Car ID', 'Price']).values
y = train_ds['Price'].values
print(f"X shape: {X.shape}, y shape: {y.shape}")


X shape: (2250, 44), y shape: (2250,)


In [978]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
    X,                 # ✓ Features only (no Car ID, no Price)
    y,                 # ✓ Price only
    test_size=0.2,
    random_state=42
)


In [979]:
print(f"The X train shape = {X_train.shape}\nThe X test  shape = {X_test.shape}")
print(f"The Y train shape = {y_train.shape}\nThe Y test  shape = {y_test.shape}")

The X train shape = (1800, 44)
The X test  shape = (450, 44)
The Y train shape = (1800,)
The Y test  shape = (450,)


feature scaling the x test and x train dat


we have seprated 80 20 data for test and train 

In [980]:
#the x _train is ndarray and we need to convert it to data frame to perform scaling
X_train=pd.DataFrame(X_train,columns=train_ds.drop(columns=['Car ID', 'Price']).columns.tolist())
# we have to convert x_test to data frame as well for scaling
X_test=pd.DataFrame(X_test,columns=train_ds.drop(columns=['Car ID', 'Price']).columns.tolist())


# Just for scaling the numeric data

In [981]:
numeric_train=X_train[['Year','Mileage','Engine Size']]
numeric_test=X_test[['Year','Mileage','Engine Size']]
numeric_train

,Year,Mileage,Engine Size
0,2022.0,62471.0,2.9
1,2003.0,231946.0,3.0
2,2005.0,5310.0,2.7
3,2010.0,44498.0,2.5
4,2013.0,93034.0,2.8
...,...,...,...
1795,2017.0,30921.0,2.7
1796,2008.0,262390.0,4.0
1797,2020.0,156546.0,3.4
1798,2023.0,102979.0,3.4


In [982]:
numeric_test

,Year,Mileage,Engine Size
0,2005.0,186869.0,3.8
1,2019.0,40934.0,1.9
2,2007.0,9304.0,2.6
3,2003.0,7887.0,3.9
4,2012.0,132994.0,3.7
...,...,...,...
445,2002.0,141280.0,4.9
446,2001.0,21711.0,4.3
447,2022.0,94626.0,3.3
448,2004.0,164164.0,6.0


# just for learning purpose i used this

In [983]:
def feature_scaling(column):
    mean_col = column.mean()
    std_col = column.std()
    
    if std_col == 0: # to avoid division by zero
        return np.zeros(len(column))
    
    return np.divide(np.subtract(column, mean_col), std_col)

In [984]:
# # Scale only numeric columns
numeric_train= numeric_train.apply(feature_scaling)
numeric_test= numeric_test.apply(feature_scaling)
numeric_train

,Year,Mileage,Engine Size
0,1.489632,-0.980357,-0.406451
1,-1.233557,0.928186,-0.336441
2,-0.946905,-1.624076,-0.546473
3,-0.230277,-1.182760,-0.686494
4,0.199701,-0.636172,-0.476462
...,...,...,...
1795,0.773004,-1.335658,-0.546473
1796,-0.516928,1.271032,0.363667
1797,1.202981,0.079069,-0.056398
1798,1.632958,-0.524176,-0.056398


In [985]:
numeric_test

,Year,Mileage,Engine Size
0,-0.923115,0.394945,0.206728
1,1.076755,-1.310903,-1.125242
2,-0.637419,-1.680629,-0.634516
3,-1.208810,-1.697193,0.276832
4,0.076820,-0.234805,0.136624
...,...,...,...
445,-1.351658,-0.137949,0.977868
446,-1.494506,-1.535602,0.557246
447,1.505299,-0.683292,-0.143790
448,-1.065963,0.129544,1.749009


# concat new feature with X train
and droppin old feature

In [986]:
X_train.drop(columns=['Year','Mileage','Engine Size'],inplace=True)
X_test.drop(columns=['Year','Mileage','Engine Size'],inplace=True)
#we will change dtypt of float to int for the one hot encoded columns
X_train=X_train.astype(int)
X_test=X_test.astype(int)
X_train

,Brand_Audi,Brand_BMW,Brand_Ford,Brand_Honda,Brand_Mercedes,Brand_Tesla,Brand_Toyota,Fuel Type_Diesel,Fuel Type_Electric,Fuel Type_Hybrid,...,Model_Model S,Model_Model X,Model_Model Y,Model_Mustang,Model_Prius,Model_Q5,Model_Q7,Model_RAV4,Model_X3,Model_X5
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,1,0,0,1,0,...,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1795,0,0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1796,0,0,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1797,0,0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1798,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [987]:
X_test

,Brand_Audi,Brand_BMW,Brand_Ford,Brand_Honda,Brand_Mercedes,Brand_Tesla,Brand_Toyota,Fuel Type_Diesel,Fuel Type_Electric,Fuel Type_Hybrid,...,Model_Model S,Model_Model X,Model_Model Y,Model_Mustang,Model_Prius,Model_Q5,Model_Q7,Model_RAV4,Model_X3,Model_X5
0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,0
446,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
447,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
448,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [988]:
X_train=pd.concat([numeric_train,X_train],axis=1 )
X_test=pd.concat([numeric_test,X_test],axis=1 )

In [989]:
print(X_train.head()[1:3])



       Year   Mileage  Engine Size  Brand_Audi  Brand_BMW  Brand_Ford  \
1 -1.233557  0.928186    -0.336441           0          0           0   
2 -0.946905 -1.624076    -0.546473           0          0           0   

   Brand_Honda  Brand_Mercedes  Brand_Tesla  Brand_Toyota  ...  Model_Model S  \
1            0               0            1             0  ...              0   
2            0               1            0             0  ...              0   

   Model_Model X  Model_Model Y  Model_Mustang  Model_Prius  Model_Q5  \
1              1              0              0            0         0   
2              0              0              0            0         0   

   Model_Q7  Model_RAV4  Model_X3  Model_X5  
1         0           0         0         0  
2         0           0         0         0  

[2 rows x 44 columns]


In [990]:
print(X_train.head()[1:3])


       Year   Mileage  Engine Size  Brand_Audi  Brand_BMW  Brand_Ford  \
1 -1.233557  0.928186    -0.336441           0          0           0   
2 -0.946905 -1.624076    -0.546473           0          0           0   

   Brand_Honda  Brand_Mercedes  Brand_Tesla  Brand_Toyota  ...  Model_Model S  \
1            0               0            1             0  ...              0   
2            0               1            0             0  ...              0   

   Model_Model X  Model_Model Y  Model_Mustang  Model_Prius  Model_Q5  \
1              1              0              0            0         0   
2              0              0              0            0         0   

   Model_Q7  Model_RAV4  Model_X3  Model_X5  
1         0           0         0         0  
2         0           0         0         0  

[2 rows x 44 columns]


In [991]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)



# sacling Y feature also
because the price ranged varies largely

In [992]:
y_test=pd.DataFrame(y_test,columns=['Price'])
y_train=pd.DataFrame(y_train,columns=['Price'])

In [993]:
# Normalize target variable (Price)
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test= y_scaler.transform(y_test.values.reshape(-1, 1)).flatten()


In [994]:
y_train

array([ 1.66058999,  0.23549326, -0.89874818, ..., -1.03377859,
       -0.19296783,  1.21377272], shape=(1800,))

In [995]:
y_test

array([ 3.51248893e-01,  2.97121025e-02, -6.96478596e-01,  1.19851353e+00,
       -1.29454510e+00,  9.83306488e-01,  2.07778633e-01,  4.12986135e-01,
       -6.63360262e-01,  1.64127014e+00,  1.50954127e+00, -5.37659340e-01,
        3.00331897e-01,  1.08977883e+00, -7.73113242e-01, -1.55489301e+00,
       -1.37540865e+00,  1.11013492e+00,  2.60935262e-01, -1.11804313e-02,
       -9.06733518e-01, -1.00257683e-01, -1.55703074e+00,  3.09507791e-01,
       -1.54794746e-01,  1.60881246e-01, -1.00454497e+00,  2.34525928e-01,
       -1.54051391e+00,  1.31429995e+00,  9.93224651e-01,  2.17722454e-01,
       -7.34815896e-01,  4.98405565e-01,  1.21030074e+00, -2.16307292e-01,
        1.32253234e+00,  1.43210498e+00, -1.53394517e-01, -2.62347192e-02,
       -9.42403786e-01,  1.51741884e+00, -1.08366520e+00, -9.67061735e-01,
       -9.75965648e-01,  1.50447479e+00, -1.63711097e+00,  4.04674937e-01,
       -1.73680467e+00,  5.25668965e-01,  1.52558892e+00,  1.00671193e+00,
       -7.05374811e-01,  

In [996]:
#our data set is finally ready for model training and testing and we 
# will now split the data set into training and testing data set and 
# then apply linear regression model on it and check the performance of 
# model using r2 score and mean absolute error

In [997]:
def compute_cost(x, y, w, b, lambda_=1):
    m = x.shape[0]
    f_wb = x @ w + b
    
    # Cost = (1/2m) * [sum of squared errors + regularization term]
    cost = np.sum(np.square(f_wb - y)) + lambda_ * np.sum(np.square(w))
    cost = cost / (2 * m)
    
    return cost

# Regularized Linear Regression - Mathematical Formulas

## Cost Function

$$J(\mathbf{w}, b) = \frac{1}{2m} \left[ \sum_{i=1}^{m} \left( f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)} \right)^2 + \lambda \sum_{j=1}^{n} w_j^2 \right]$$

where:
- $f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b = \sum_{j=1}^{n} w_j x_j + b$
- $m$ = number of training examples
- $n$ = number of features
- $\lambda$ = regularization parameter
- $b$ is not regularized

---

## Gradient Computation

### Partial derivative with respect to $w_j$:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} \left( f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)} \right) x_j^{(i)} + \frac{\lambda}{m} w_j$$

for $j = 1, 2, \ldots, n$

### Partial derivative with respect to $b$:

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} \left( f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)} \right)$$

---

## Gradient Descent Algorithm

**Repeat until convergence:**

$$w_j := w_j - \alpha \frac{\partial J}{\partial w_j} \quad \text{for } j = 1, 2, \ldots, n$$

$$b := b - \alpha \frac{\partial J}{\partial b}$$

where $\alpha$ is the learning rate

---

## Gradient Descent (Expanded Form)

### Weight updates:

$$w_j := w_j - \frac{\alpha}{m} \left[ \sum_{i=1}^{m} \left( f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)} \right) x_j^{(i)} + \lambda w_j \right]$$

### Bias update:

$$b := b - \frac{\alpha}{m} \sum_{i=1}^{m} \left( f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)} \right)$$

---

## Vector Notation (Matrix Form)

### Gradient:

$$\nabla_{\mathbf{w}} J = \frac{1}{m} \mathbf{X}^T (\mathbf{X}\mathbf{w} + b\mathbf{1} - \mathbf{y}) + \frac{\lambda}{m} \mathbf{w}$$

$$\frac{\partial J}{\partial b} = \frac{1}{m} \mathbf{1}^T (\mathbf{X}\mathbf{w} + b\mathbf{1} - \mathbf{y})$$

### Update Rules:

$$\mathbf{w} := \mathbf{w} - \alpha \nabla_{\mathbf{w}} J$$

$$b := b - \alpha \frac{\partial J}{\partial b}$$

where:
- $\mathbf{X}$ is the $m \times n$ feature matrix
- $\mathbf{y}$ is the $m \times 1$ target vector
- $\mathbf{1}$ is an $m \times 1$ vector of ones
- $\mathbf{w}$ is the $n \times 1$ weight vector

---

## Key Notes

1. **Regularization applies only to weights**, not bias: The regularization term is $\lambda \sum_{j=1}^{n} w_j^2$, which does not include $b$

2. **Simultaneous update**: In gradient descent, all parameters should be updated simultaneously (calculate all gradients first, then update)

3. **Learning rate $\alpha$**: Controls the step size. Too large may cause divergence, too small results in slow convergence

4. **Regularization parameter $\lambda$**: 
   - $\lambda = 0$ → no regularization (standard linear regression)
   - Large $\lambda$ → heavy regularization (simpler model, may underfit)
   - Small $\lambda$ → light regularization

In [1]:
def compute_gradient(x, y, w, b, lambda_=1):
    m = x.shape[0]  # Define m - number of training examples
    
    f_wb = np.dot(x, w) + b
    error = f_wb - y
    
    # Correct: Use x.T @ error, not error @ x
    # Correct: Use lambda_ * w, not lambda_ @ w
    dj_dw = (1/m) * (x.T @ error + lambda_ * w)
    dj_db = (1/m) * np.sum(error)
    
    return dj_dw, dj_db

In [2]:
def gradient_descent(x, y, w, b, alpha, num, lambda_=1, print_cost=False):
    cost_history = []
    
    for i in range(num):
        dj_dw, dj_db = compute_gradient(x, y, w, b, lambda_)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        
        # Track and optionally print cost
        if i % 100 == 0:
            cost = compute_cost(x, y, w, b, lambda_)
            cost_history.append(cost)
            if print_cost:
                print(f"Iteration {i:4d}: Cost {cost:0.6f}")
    
    return w, b, cost_history